# Smoke Test — Gemma 2 9B (Phase B.1)

**Goal:** Check whether the LSI infrastructure developed for Gemma 2 2B generalizes to Gemma 2 9B. A go / no-go decision before replicating the full pipeline.

**Setup:**
- Model: `google/gemma-2-9b` (42 layers, $d_\text{model} = 3584$)
- SAE: `gemma-scope-9b-pt-res-canonical`, layer 20 (~48% depth), width 16k
- Corpus: 20 PT prompts + 20 EN prompts, comparable lengths (Wikipedia-style)
- Expected VRAM: ~18 GB in bfloat16 (fits on Colab L4 with headroom)

**Pass / fail criterion:**
- **PASS** if ≥ 30 active features with $\text{LSI} > 0{,}3$ (PT-dominant) AND the top-1 PT feature is visually coherent (active only on PT tokens).
- **FAIL** if < 10 PT-dominant features OR if top-1 activates similarly in PT and EN.
- **YELLOW** between 10 and 30 → revise prompts / increase tokens / try another layer.

**Comparison:** on Gemma 2B layer 13 with ~1M tokens per language, we observed 651 PT-both features ($|\text{LSI}| > 0{,}3$ on both datasets). Here we use ~2 orders of magnitude fewer tokens, so this is a "signal present" test, not a magnitude test.

## 1. Installation and environment

In [ ]:
!pip install -q sae-lens transformer-lens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:    {name} ({total_gb:.1f} GB total)")
    assert total_gb >= 20, "Gemma 2 9B precisa de >=20GB VRAM em bf16. Use L4 ou A100."
else:
    raise RuntimeError("Smoke test exige GPU.")

## 2. Configuration

In [ ]:
MODEL_NAME = "gemma-2-9b"
LAYER = 20
WIDTH = "16k"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_{WIDTH}/canonical"
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

DTYPE = torch.bfloat16
MIN_ACTS = 2
THRESH_PT = 0.30
THRESH_EN = -0.30
THRESH_CROSS = 0.10

print(f"Model:    {MODEL_NAME}")
print(f"SAE:      {SAE_RELEASE} :: {SAE_ID}")
print(f"Hook:     {HOOK_NAME}")
print(f"dtype:    {DTYPE}")

## 3. Load model and SAE

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.empty_cache()
gc.collect()

HF_MODEL_ID = f"google/{MODEL_NAME}"
print(f"Loading Gemma 2 9B directly in {DTYPE} (~4-6 min na L4)...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": device},
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
depth_pct = 100 * LAYER / n_layers
print(f"Model: {HF_MODEL_ID} | layers: {n_layers} | d_model: {d_model}")
print(f"Layer {LAYER} corresponde a {depth_pct:.1f}% da profundidade.")
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
from sae_lens import SAE

print(f"Loading SAE {SAE_ID}...")
sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)
if isinstance(sae, tuple):
    sae = sae[0]
sae = sae.to(DTYPE)
_hook_meta = getattr(sae.cfg, 'hook_name', None) or getattr(getattr(sae.cfg, 'metadata', None), 'hook_name', 'n/a')
print(f"SAE: d_sae={sae.cfg.d_sae} | d_in={sae.cfg.d_in} | hook={_hook_meta}")
assert sae.cfg.d_in == d_model, "d_in of the SAE does not match d_model of the model."
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
target_layer = model.model.layers[LAYER]
_cap = {}

def _capture_hook(module, inputs, output):
    h = output[0] if isinstance(output, tuple) else output
    _cap["resid"] = h

def encode_text(text):
    """Tokenizes, runs forward, returns feature_activations (n_tokens, d_sae) and str_tokens."""
    enc = tokenizer(text, return_tensors="pt").to(device)
    str_tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    handle = target_layer.register_forward_hook(_capture_hook)
    try:
        with torch.no_grad():
            model(**enc, use_cache=False)
        resid = _cap["resid"]
        feat = sae.encode(resid)[0]
    finally:
        handle.remove()
        _cap.clear()
    return feat, str_tokens

## 4. Sanity check (1 prompt PT + 1 EN)

In [ ]:
sanity_pairs = [
    ("PT", "O Brasil é o maior país da América do Sul e tem uma população diversa."),
    ("EN", "Brazil is the largest country in South America with a diverse population."),
]
for lang, text in sanity_pairs:
    feat, _ = encode_text(text)
    l0 = (feat > 0).float().sum(dim=-1).mean().item()
    print(f"[{lang}] {text[:55]}")
    print(f"      mean L0: {l0:.1f} features/token | n_tokens={feat.shape[0]}")
    print()

## 5. Smoke test corpus

20 PT prompts + 20 EN prompts, a mix of domains (Wikipedia, encyclopedic description, colloquial narrative) to avoid overfitting to a single register.

In [ ]:
PROMPTS_PT = [
    "O Brasil é o maior país da América do Sul, com mais de 200 milhões de habitantes e uma cultura formada por diferentes povos indígenas, africanos e europeus.",
    "A capital do Brasil é Brasília, inaugurada em 1960 como parte de um plano de interiorização do país, e foi projetada pelos arquitetos Oscar Niemeyer e Lúcio Costa.",
    "Machado de Assis é considerado um dos maiores escritores da literatura em língua portuguesa, autor de romances como Memórias Póstumas de Brás Cubas e Dom Casmurro.",
    "A floresta amazônica cobre cerca de 5,5 milhões de quilômetros quadrados, sendo a maior floresta tropical do mundo e abrigando uma enorme biodiversidade.",
    "O futebol foi trazido ao Brasil no final do século XIX por Charles Miller, e desde então se tornou o esporte mais popular do país.",
    "A bossa nova surgiu no Rio de Janeiro no fim dos anos 1950, misturando elementos do samba com a harmonia do jazz e tendo como nomes centrais João Gilberto, Tom Jobim e Vinícius de Moraes.",
    "A culinária mineira é conhecida pelo uso do feijão tropeiro, do tutu, do queijo curado da serra da Canastra e dos doces preparados em panela de cobre.",
    "Durante o Império, Dom Pedro II governou o Brasil por quase cinquenta anos, período no qual o país consolidou suas fronteiras e iniciou seu processo de industrialização.",
    "A menina bonita foi à praia com sua avó e ficou impressionada com a quantidade de barracas, vendedores de água de coco e crianças correndo na areia.",
    "O médico explicou que era preciso tomar o remédio três vezes ao dia, sempre depois das refeições, e voltar à consulta na semana seguinte.",
    "A advogada chegou cedo ao escritório, organizou os processos da manhã e telefonou para a cliente que aguardava notícias sobre a audiência.",
    "O carnaval de Salvador ocupa as ruas durante quase uma semana, com trios elétricos, blocos afro e milhares de pessoas dançando atrás do caminhão.",
    "A reforma agrária ainda é tema de debate no Brasil, envolvendo movimentos sociais, parlamentares e proprietários de terras em discussões muitas vezes acaloradas.",
    "O Pão de Açúcar, no Rio de Janeiro, é um dos pontos turísticos mais visitados do país, com bondinho que liga a praia da Urca ao morro da Urca e ao topo do Pão.",
    "Os bandeirantes percorreram o sertão em busca de ouro, índios e pedras preciosas, e acabaram contribuindo para a expansão territorial do Brasil colonial.",
    "Em São Paulo, a avenida Paulista concentra museus, sedes de bancos, hospitais e teatros, e nos domingos é fechada para carros, virando ponto de encontro popular.",
    "A engenheira responsável pela obra explicou aos moradores que o cronograma havia atrasado por causa das chuvas e que a entrega seria adiada em dois meses.",
    "O presidente discursou em frente ao Congresso Nacional, prometendo reformas econômicas, redução da inflação e novos investimentos em infraestrutura.",
    "A festa junina é uma das tradições mais queridas do nordeste, com quadrilha, comidas típicas como pamonha e canjica, e fogueira no meio do arraial.",
    "O escritor português José Saramago, vencedor do prêmio Nobel de literatura em 1998, é autor de obras como Ensaio sobre a Cegueira e Memorial do Convento.",
]

PROMPTS_EN = [
    "The United States is the third largest country by area in the world, with over 330 million people and a culture shaped by waves of immigration from many regions.",
    "The capital of the United States is Washington, D.C., founded in 1790 along the Potomac River, and home to many federal institutions and national monuments.",
    "William Shakespeare is widely regarded as one of the greatest writers in the English language, the author of plays such as Hamlet, Macbeth, and King Lear.",
    "The Amazon rainforest covers about 5.5 million square kilometers, making it the largest tropical forest in the world and home to an enormous biodiversity.",
    "Baseball became popular in the United States during the nineteenth century, and since then it has been considered the country's national pastime.",
    "Jazz emerged in New Orleans in the late nineteenth and early twentieth centuries, blending African American musical traditions with European harmonic structures, and producing artists like Louis Armstrong and Duke Ellington.",
    "Southern cuisine in the United States is known for its use of slow-cooked beans, smoked meats, aged cheeses from the Appalachian region, and desserts prepared in copper pans.",
    "During the nineteenth century, Queen Victoria ruled the United Kingdom for nearly seventy years, a period in which the country consolidated its borders and accelerated its industrial revolution.",
    "The pretty girl went to the beach with her grandmother and was impressed by the number of stalls, coconut-water vendors, and children running on the sand.",
    "The doctor explained that it was necessary to take the medicine three times a day, always after meals, and to come back for a check-up the following week.",
    "The lawyer arrived at the office early, organized the morning's cases, and called the client who was waiting for news about the hearing.",
    "The carnival of New Orleans takes over the streets for nearly a week, with marching bands, krewes, and thousands of people dancing behind the floats.",
    "Land reform is still a topic of debate in many countries, involving social movements, lawmakers, and landowners in often heated discussions.",
    "The Statue of Liberty, in New York City, is one of the most visited landmarks in the country, with a ferry connecting Battery Park to Liberty Island and to the pedestal of the statue.",
    "American frontiersmen traveled the wilderness in search of gold, indigenous tribes, and precious stones, and ended up contributing to the territorial expansion of the colonial United States.",
    "In New York, Fifth Avenue concentrates museums, bank headquarters, hospitals, and theaters, and on some Sundays it is closed to cars, becoming a popular gathering point.",
    "The engineer in charge of the project explained to the residents that the schedule had been delayed due to heavy rains and that delivery would be postponed by two months.",
    "The president addressed the public in front of the Capitol, promising economic reforms, reduction of inflation, and new investments in infrastructure.",
    "Thanksgiving is one of the most beloved holidays in the United States, with family gatherings, traditional foods such as turkey and pumpkin pie, and a parade in New York.",
    "The Irish writer Seamus Heaney, winner of the Nobel Prize in Literature in 1995, is the author of works such as Death of a Naturalist and The Spirit Level.",
]

print(f"Prompts: PT={len(PROMPTS_PT)} | EN={len(PROMPTS_EN)}")
tok_pt = sum(len(tokenizer(p)["input_ids"]) for p in PROMPTS_PT)
tok_en = sum(len(tokenizer(p)["input_ids"]) for p in PROMPTS_EN)
print(f"Tokens:  PT≈{tok_pt} | EN≈{tok_en} | total≈{tok_pt + tok_en}")

## 6. Activation extraction and per-feature count

In [ ]:
from tqdm.auto import tqdm

d_sae = sae.cfg.d_sae

def feature_counts(prompts):
    """Counts tokens where each feature is active (>0) and total tokens."""
    counts = torch.zeros(d_sae, dtype=torch.float32, device=device)
    total_tokens = 0
    for p in tqdm(prompts, desc="prompts"):
        feat, _ = encode_text(p)
        per_feat = (feat > 0).float().sum(dim=0)
        counts += per_feat
        total_tokens += feat.shape[0]
        del feat
        torch.cuda.empty_cache()
    return counts.cpu(), total_tokens

print("PT...")
counts_pt, tok_pt_actual = feature_counts(PROMPTS_PT)
print("EN...")
counts_en, tok_en_actual = feature_counts(PROMPTS_EN)

freq_pt = counts_pt / max(tok_pt_actual, 1)
freq_en = counts_en / max(tok_en_actual, 1)
print(f"tokens PT={tok_pt_actual} | tokens EN={tok_en_actual}")

## 7. LSI and global statistics

In [ ]:
eps = 1e-10
lsi = (freq_pt - freq_en) / (freq_pt + freq_en + eps)
active = (counts_pt + counts_en) >= MIN_ACTS
n_active = active.sum().item()
n_total = d_sae

lsi_a = lsi[active]
n_pt = (lsi_a > THRESH_PT).sum().item()
n_en = (lsi_a < THRESH_EN).sum().item()
n_cross = (lsi_a.abs() < THRESH_CROSS).sum().item()

print(f"Total features:          {n_total}")
print(f"Active features (>={MIN_ACTS}):   {n_active} ({100*n_active/n_total:.1f}%)")
print()
print(f"About active features:")
print(f"  mean LSI:               {lsi_a.mean().item():+.4f}")
print(f"  LSI mediano:            {lsi_a.median().item():+.4f}")
print(f"  PT-dominantes (>{THRESH_PT:+.2f}):  {n_pt}")
print(f"  EN-dominantes (<{THRESH_EN:+.2f}): {n_en}")
print(f"  Cross-lingual (|LSI|<{THRESH_CROSS:.2f}): {n_cross}")

## 8. Top features PT-dominantes e EN-dominantes

In [ ]:
TOP_K = 10

lsi_pt = lsi.clone()
lsi_pt[~active] = -float("inf")
top_pt_vals, top_pt_idx = lsi_pt.topk(TOP_K)

lsi_en = lsi.clone()
lsi_en[~active] = float("inf")
top_en_vals, top_en_idx = (-lsi_en).topk(TOP_K)

print(f"Top-{TOP_K} PT-dominantes:")
print(f"  {'feat':>6}  {'LSI':>7}  {'freq_PT':>9}  {'freq_EN':>9}  {'count_PT':>9}  {'count_EN':>9}")
for i in top_pt_idx.tolist():
    print(f"  {i:>6}  {lsi[i].item():>+7.3f}  {freq_pt[i].item():>9.5f}  {freq_en[i].item():>9.5f}  "
          f"{int(counts_pt[i].item()):>9}  {int(counts_en[i].item()):>9}")

print()
print(f"Top-{TOP_K} EN-dominantes:")
print(f"  {'feat':>6}  {'LSI':>7}  {'freq_PT':>9}  {'freq_EN':>9}  {'count_PT':>9}  {'count_EN':>9}")
for i in top_en_idx.tolist():
    print(f"  {i:>6}  {lsi[i].item():>+7.3f}  {freq_pt[i].item():>9.5f}  {freq_en[i].item():>9.5f}  "
          f"{int(counts_pt[i].item()):>9}  {int(counts_en[i].item()):>9}")

## 9. Qualitative visualization: top-1 PT-dominant token by token

Checks whether the top-1 PT feature actually fires on Portuguese tokens and stays silent on English ones. If it activates similarly in both, it is a yellow flag.

In [ ]:
feat_id = int(top_pt_idx[0].item())
print(f"Focus feature: {feat_id} (LSI = {lsi[feat_id].item():+.3f})")
print()

def trace_feature(prompt, fid, lang):
    feat, str_tokens = encode_text(prompt)
    vals = feat[:, fid].float().cpu().tolist()
    fire = [(t, v) for t, v in zip(str_tokens, vals) if v > 0]
    print(f"[{lang}] {prompt[:80]}{'…' if len(prompt) > 80 else ''}")
    print(f"      active tokens ({len(fire)}/{len(vals)}): ", end="")
    print(", ".join(f"{repr(t)}={v:.2f}" for t, v in fire[:8]) + (" …" if len(fire) > 8 else ""))
    print()

for p in PROMPTS_PT[:3]:
    trace_feature(p, feat_id, "PT")
for p in PROMPTS_EN[:3]:
    trace_feature(p, feat_id, "EN")

## 10. Veredicto go / no-go

In [ ]:
if n_pt >= 30:
    verdict = "PASS"
    msg = "Sinal de features PT-dominantes presente. Seguir com replicação do pipeline completo no 9B."
elif n_pt >= 10:
    verdict = "YELLOW"
    msg = ("Sinal presente mas fraco. Antes de ir adiante: "
           "(i) aumentar tokens (200-500 prompts), (ii) testar layer 24 ou 28, "
           "(iii) revisar prompts em PT (mais variedade morfossintática).")
else:
    verdict = "FAIL"
    msg = ("Quase nenhum sinal PT-dominante. Hipóteses: "
           "(i) layer 20 não tem features de idioma neste SAE "
           "(testar layer 9, 24, 28); "
           "(ii) MIN_ACTS muito alto; "
           "(iii) verificar se sae.encode aplica JumpReLU corretamente.")

print("=" * 60)
print(f"VEREDICTO: {verdict}")
print("=" * 60)
print(f"PT-dominantes (LSI > {THRESH_PT}): {n_pt}")
print(f"EN-dominantes (LSI < {THRESH_EN}): {n_en}")
print(f"Cross-lingual (|LSI| < {THRESH_CROSS}): {n_cross}")
print()
print(msg)

## 11. Save result

In [ ]:
import json

result = {
    "experiment": "gemma9b_smoke_test",
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "sae_id": SAE_ID,
    "layer": LAYER,
    "depth_pct": round(depth_pct, 2),
    "width": WIDTH,
    "d_sae": d_sae,
    "n_prompts_pt": len(PROMPTS_PT),
    "n_prompts_en": len(PROMPTS_EN),
    "tokens_pt": int(tok_pt_actual),
    "tokens_en": int(tok_en_actual),
    "min_acts": MIN_ACTS,
    "thresholds": {"pt": THRESH_PT, "en": THRESH_EN, "cross": THRESH_CROSS},
    "summary": {
        "n_active": int(n_active),
        "n_pt_dominant": int(n_pt),
        "n_en_dominant": int(n_en),
        "n_cross_lingual": int(n_cross),
        "lsi_mean_active": float(lsi_a.mean().item()),
        "lsi_median_active": float(lsi_a.median().item()),
    },
    "top10_pt": [
        {
            "feature_id": int(i),
            "lsi": float(lsi[i].item()),
            "freq_pt": float(freq_pt[i].item()),
            "freq_en": float(freq_en[i].item()),
        }
        for i in top_pt_idx.tolist()
    ],
    "top10_en": [
        {
            "feature_id": int(i),
            "lsi": float(lsi[i].item()),
            "freq_pt": float(freq_pt[i].item()),
            "freq_en": float(freq_en[i].item()),
        }
        for i in top_en_idx.tolist()
    ],
    "verdict": verdict,
}

out_path = f"gemma9b_smoke_test_layer{LAYER}.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"Saved to {out_path}")
print(f"VEREDICTO: {verdict} | n_pt={n_pt} | n_en={n_en} | n_cross={n_cross}")

## 12. Next steps according to the verdict

**If PASS:**
1. Replicate the full pipeline on 9B layer 20 (LSI with 1M+ tokens, PT-Wiki vs FineWeb triangulation).
2. Run the reduced version of `exp_probes_multilayer.ipynb` on 9B: 3 phenomena (gender, crase, clitics) across 3 layers (e.g., 9, 20, 31) with the gender benchmark.
3. Add a "Cross-scale generalization: 2B → 9B" section to the paper.

**If YELLOW:**
- Increase the corpus to ~500 prompts or ~50k tokens per language.
- Try the SAE on another layer (suggestions: 9 and 28, to sweep 21%/67% depth).
- Re-evaluate.

**If FAIL:**
- Check that the hook reads from the correct `resid_post`.
- Confirm a consistent `dtype` between model and SAE.
- Manually inspect the top feature: if it activates in both languages, the issue is the LSI signal, not the engineering.
- Before discarding 9B, repeat the test at layer 9 (analogous to layer 5 of 2B, where there was greater cross-linguality).